# MCP核心概念

三个核心概念：
|概念|说明|
|----|----|
|MCP Server|提供MCP服务的应用，可以是远程服务，也可以是本地服务|
|MCP Client|连接到MCP服务器，读取MCP信息（特别是Tool信息），供Host使用|
|MCP Host|协调和管理多个MCP Client的AI应用，比如LangChain的Agent|

Server 提供的是"工具能力"，Client 负责搬运工具定义和调用结果，Host（里的模型）负责决策。

![alt text](../mcp-concepts.png)

MCP Client 与 MCP Server之间有两种通信协议：
- stdio
- streamable_http

stdio就是标准输入输出，MCP Client运行时，分两种情况：
- 外部服务：Client会把这个MCP服务的脚本下载到本地，然后作为一个子进程运行。
- 本地服务：Client会把本地脚本直接加载，作为一个子进程运行
也就是说，stdio模式中，MCP Client 和 MCP Server之间的通信就是进程通信，没有网络延迟。

streamable_http其实就是可以用event stream来发送数据的http模式，本质还是Server Event Stream，也就是SSE。也就是说MCP client通过发送http请求与MCP server交互。因此存在一定的网络延迟。

# 连接外部MCP服务

可以在https://mcp.so/zh/搜索各种MCP服务

In [ ]:
# pip install langchain-mcp-adapters
# 安装MCP依赖库

## Time MCP服务

MCP脚本下载的方式取决于配置中的command，常见的有两种：
- npx : 基于node.js的包管理工具
- uvx : 基于uv(python的uv工具)的包管理工具  
因此，本地环境必须支持npx、uvx命令

在LangChain中除了基本的服务器配置，还必须设置一个transport参数，指定MCP服务的通信方式，有两个可选值：
- stdio
- http或者streamable_http

In [ ]:
from langchain_mcp_adapters.client import MultiServerMCPClient

# 连接Time MCP服务器
client = MultiServerMCPClient(
    {
        "time": {
            "transport": "stdio",
            "command": "npx",
            "args": [
                "mcp-server-time",
                "--local-timezone=Asia/Shanghai"
            ]
        }
    }
)

# 注意MultiServerMCPClient是异步的，结果都是协程对象，需要await
tools = await client.get_tools()

In [ ]:
for tool in tools:
    print(tool.name)
    print(tool.description)
    print("------------------------")

In [ ]:
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage

agent = create_agent("deepseek-v4-flash", tools)

response = await agent.ainvoke(
    {"messages": [HumanMessage("现在几点了？")]}
)

for message in response["messages"]:
    message.pretty_print()

## Deepwiki MCP Server

In [9]:
from langchain_mcp_adapters.client import MultiServerMCPClient
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage

# 连接Deepwiki MCP Server
client = MultiServerMCPClient(
    {
        "deepwiki": {                       # server 名
            "url": "https://mcp.deepwiki.com/mcp",  # 官方托管端点
            "transport": "http",
        }
    }
)

tools = await client.get_tools()

In [12]:
agent = create_agent(
    model="deepseek-v4-flash",
    tools=tools,
    system_prompt="You are a travel agent. Help user find best flights. No follow up questions. If the user uses Chinese, use zh-cn as the locale value."
)

response = await agent.ainvoke(
    {"messages": [HumanMessage("查一下2026年5月15日晚上从伦敦飞纽约的航班。")]}
)

for message in response["messages"]:
    message.pretty_print()

================================ Human Message =================================

查一下2026年5月15日晚上从伦敦飞纽约的航班。
================================== Ai Message ==================================

很抱歉，我当前的工具并不支持实时航班搜索（我只能查询 GitHub 仓库的文档和问答），因此无法为您查询 2026 年 5 月 15 日伦敦飞纽约的具体航班和票价。

不过，我可以为您提供一些实用的参考信息，帮助您找到最佳航班：

**主要航线信息（伦敦 → 纽约）**

- **主要出发机场**：伦敦希思罗（LHR）、盖特威克（LGW）、斯坦斯特德（STN）
- **主要到达机场**：纽约肯尼迪（JFK）、纽瓦克（EWR）
- **飞行时长**：约 7 至 8 小时（顺风时更快）
- **时差**：纽约比伦敦晚 5 小时（EDT），因此晚上的航班到达后通常是当地的傍晚或夜晚

**提供伦敦—纽约晚间直飞航班的常见航空公司**
- **British Airways（英航）**：LHR → JFK，晚间有多班，是体验较舒适的航班之一
- **Virgin Atlantic（维珍航空）**：LHR → JFK，以服务出色著称
- **American Airlines（美国航空）**：LHR → JFK/EWR
- **Delta（达美航空）**：LHR → JFK
- **United（美联航）**：LHR → EWR
- **JetBlue**：LHR/LGW → JFK，经济舱体验口碑不错

**建议查询方式**
1. 使用 Google Flights、Skyscanner、Kayak 或 Expedia 等比价平台，输入日期 **2026-05-15 晚间**（约 18:00–23:00 出发）。
2. 直接在航空公司官网确认航班时刻表（2026 年夏秋季时刻表通常会在 2026 年春季发布）。
3. 关注是否有中转航班更便宜，但直飞通常更方便。

如果您能告诉我是否有其他查询需求（比如偏好直飞、预算范围、舱位等），我可以尽力提供更多选座、行李、过境方面的建议。祝您出行顺利！✈️


# 自定义MCP服务

## 创建简单的MCP服务器
pip install fastmcp

只需要定义几个方法，然后利用FastMCP提供的装饰器即可：
- @mcp.tool : 作为MCP中的工具
- @mcp.resources : 返回MCP需要的resources，类似拓展知识库
- @mcp.prompt : 返回MCP预定义的Prompt，预设的提示词  

通常我们只需要定义带有tool的MCP Server就可以了。

由于采用stdio方式，因此这个文件写好放在那里，不需要启动，将来MCP Client会自己启动并加载为子进程。

## 连接自定义MCP服务

由于我们自定义的MCP是本地py文件，所以启动的Command直接就是python，而不是npx或uvx

In [ ]:
from langchain_mcp_adapters.client import MultiServerMCPClient
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage

# 连接自定义MCP服务
client = MultiServerMCPClient(
    {
        "math": {
            "transport": "stdio",
            "command": "python",
            "args": [r"C:\Users\86180\Vscode_Project\agent_learning\Agent进阶\math_mcp_server.py"],  # 脚本绝对路径
        }
    }
)


tools = await client.get_tools()

# 创建agent
agent = create_agent(
    model="deepseek-chat",
    tools=tools,
    system_prompt="You are a helpful agent. You must use tools to answer math question."
)

# 调用测试
response = await agent.ainvoke({
    "messages": [HumanMessage("467和529的平方根之和是多少?")]
})

for message in response["messages"]:
    message.pretty_print()